# Experimentos completos — Trahan + Black–Scholes + mercado + losses

Este notebook junta as análises principais:

1. **Mercado real primeiro**: descrição do contrato NDX escolhido.
2. **Seleção de modelos já rodados**: filtros por `hidden`, `blocks`, `n_qubits`, `n_layers`, `seed`, parâmetros etc.
3. **Losses**: curvas completas, melhores runs e análise “anytime” por checkpoints.
4. **Teste sintético de 1 dólar**: taxa de sucesso `MAE_V_global < 1`.
5. **Comparação por orçamento de parâmetros**, estilo Trahan.
6. **Curvas sintéticas 1D**: preço e Greeks contra Black–Scholes.
7. **Mercado real 1D** com normalização correta por strike.
8. **Resumo final para artigo**.

Não usa `CQNN`. Não compara tempo de cálculo.

## 0. Setup

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import json
import math
import warnings
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import norm

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "experimentos_pinn"
DATA_DIR = PROJECT_ROOT / "data"

OUT_DIR = PROJECT_ROOT / "figures_experimentos_completo"
TAB_DIR = PROJECT_ROOT / "tables_experimentos_completo"
OUT_DIR.mkdir(exist_ok=True)
TAB_DIR.mkdir(exist_ok=True)

# Parâmetros do problema sintético usado no treino
TRAIN_K = 40.0
TRAIN_S_MAX = 160.0
TRAIN_T = 1.0
TRAIN_V_MAX = 120.0
TRAIN_R = 0.05
TRAIN_SIGMA = 0.2

# Critério de sucesso sintético
SYNTHETIC_ONE_DOLLAR_THRESHOLD = 1.0

# Métodos do artigo: sem CQNN
FAMILIES = ["CLASSICO", "QNN", "HQNN"]

# Use False para avaliar só melhores modelos por família nas curvas.
# Use True para avaliar todos nas métricas já salvas; curvas continuam selecionadas.
EVALUATE_ALL_RUNS_FOR_CURVES = False

# Checkpoints de loss para análise "anytime".
# Não use isso como comparação principal se não tiver sido definido antes do experimento.
LOSS_CHECKPOINTS = [100, 250, 500, 1000, 2500, 5000, 10000]

DEVICE = "cpu"

### 0B. Seleção dos modelos já rodados

Mude esta célula para controlar a comparação.  
Use `None` quando não quiser filtrar uma coluna.

In [ ]:
MODEL_SELECTION = {
    "CLASSICO": {
        "model_type": ["MLP", "ResNet"],
        "hidden": None,           # ex.: [2, 3, 5, 10]
        "blocks": None,           # ex.: [1, 2, 3, 4, 5]
        "activation": None,
        "seed": None,
    },
    "QNN": {
        "model_type": ["QNN", "QPINN"],
        "n_qubits": None,         # ex.: [2, 3, 4, 7]
        "n_layers": None,         # ex.: [1, 2, 3]
        "entangler": None,
        "seed": None,
    },
    "HQNN": {
        "model_type": ["HQNN"],
        "hidden": None,           # ex.: [2, 3, 5]
        "blocks": None,           # ex.: [1, 2, 3, 5]
        "n_qubits": None,         # ex.: [2, 3, 5]
        "n_layers": None,         # ex.: [1, 2, 3]
        "entangler": None,
        "seed": None,
    },
}

# Corte opcional de parâmetros para deixar tudo mais parecido.
# Ex.: MAX_PARAMS = 400
MAX_PARAMS = None

# Faixas de parâmetros para comparação justa.
PARAM_BINS = [0, 50, 100, 200, 400, 800, 1600, np.inf]
PARAM_LABELS = ["0-50", "50-100", "100-200", "200-400", "400-800", "800-1600", "1600+"]

### 0C. Estilo gráfico

In [ ]:
COLORS = {
    "Market": "black",
    "Black-Scholes IV": "#5E81B5",
    "Black-Scholes train sigma": "#E19C24",
    "CLASSICO": "#8FB131",
    "QNN": "#80699B",
    "HQNN": "#C44E52",
}

mpl.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.04,
    "savefig.dpi": 300,
    "figure.dpi": 130,
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "axes.edgecolor": "black",
    "axes.linewidth": 0.9,
    "axes.grid": True,
    "grid.color": "0.84",
    "grid.linewidth": 0.55,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "legend.frameon": False,
})

def savefig(name: str):
    path = OUT_DIR / name
    plt.tight_layout()
    plt.savefig(path)
    plt.show()
    print(f"[OK] figura salva: {path}")
    return path

## 1. Funções auxiliares

In [ ]:
def bs_call_greeks_tau(S, tau, K, r, sigma):
    """Black-Scholes call com tau = tempo até maturidade."""
    S = np.asarray(S, dtype=float)
    tau = np.asarray(tau, dtype=float)
    K = np.asarray(K, dtype=float)
    r = np.asarray(r, dtype=float)
    sigma = np.asarray(sigma, dtype=float)

    eps = 1e-10
    S_safe = np.maximum(S, eps)
    tau_safe = np.maximum(tau, eps)
    sigma_safe = np.maximum(sigma, eps)

    d1 = (np.log(S_safe / K) + (r + 0.5 * sigma_safe**2) * tau_safe) / (
        sigma_safe * np.sqrt(tau_safe)
    )
    d2 = d1 - sigma_safe * np.sqrt(tau_safe)

    V = S_safe * norm.cdf(d1) - K * np.exp(-r * tau_safe) * norm.cdf(d2)
    delta = norm.cdf(d1)
    gamma = norm.pdf(d1) / (S_safe * sigma_safe * np.sqrt(tau_safe))

    # Theta financeiro usual: dV/dcalendar_time = - dV/dtau.
    theta = (
        -(S_safe * norm.pdf(d1) * sigma_safe) / (2 * np.sqrt(tau_safe))
        - r * K * np.exp(-r * tau_safe) * norm.cdf(d2)
    )

    payoff = np.maximum(S - K, 0.0)
    V = np.where(tau <= eps, payoff, V)
    return V, delta, gamma, theta


def bs_call_greeks_t(S, t, K=TRAIN_K, r=TRAIN_R, sigma=TRAIN_SIGMA, T=TRAIN_T):
    """Black-Scholes call usando t da PDE, com tau = T - t."""
    tau = np.maximum(T - np.asarray(t, dtype=float), 0.0)
    return bs_call_greeks_tau(S, tau, K, r, sigma)


def metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y = y_true[mask]
    p = y_pred[mask]
    if len(y) == 0:
        return pd.Series({"N": 0, "RMSE": np.nan, "MAE": np.nan, "Bias": np.nan, "Corr": np.nan})
    e = p - y
    corr = np.nan
    if len(y) > 1 and np.std(y) > 0 and np.std(p) > 0:
        corr = np.corrcoef(y, p)[0, 1]
    return pd.Series({
        "N": len(y),
        "RMSE": np.sqrt(np.mean(e**2)),
        "MAE": np.mean(np.abs(e)),
        "Bias": np.mean(e),
        "Corr": corr,
    })


def normalize_family(row_or_value):
    text = str(row_or_value)
    if isinstance(row_or_value, pd.Series):
        text = " ".join(
            str(row_or_value.get(c, ""))
            for c in ["family", "model_type", "model_label", "run_id", "source_summary"]
        )
    low = text.lower()
    if "cqnn" in low or "cquant" in low:
        return "CQNN"
    if "hqnn" in low or "hybrid" in low or "hibrid" in low:
        return "HQNN"
    if "qnn" in low or "qpinn" in low or "quant" in low:
        return "QNN"
    if "mlp" in low or "resnet" in low or "classic" in low or "classico" in low:
        return "CLASSICO"
    return str(row_or_value)


def to_numeric_if_possible(df):
    out = df.copy()
    for c in out.columns:
        if out[c].dtype == "object":
            conv = pd.to_numeric(out[c], errors="coerce")
            if conv.notna().sum() >= 0.7 * out[c].notna().sum() and out[c].notna().sum() > 0:
                out[c] = conv
    return out


def apply_model_selection(df, selection=MODEL_SELECTION, max_params=MAX_PARAMS):
    d = df.copy()
    if "family" not in d.columns:
        d["family"] = d.apply(normalize_family, axis=1)
    d["family"] = d["family"].apply(normalize_family)

    # Remover CQNN
    d = d[d["family"].isin(FAMILIES)].copy()

    parts = []
    for fam, rules in selection.items():
        sub = d[d["family"] == fam].copy()
        for col, allowed in rules.items():
            if allowed is None:
                continue
            if col in sub.columns:
                sub = sub[sub[col].isin(allowed)]
        parts.append(sub)

    if parts:
        d = pd.concat(parts, ignore_index=True)
    else:
        d = d.iloc[0:0].copy()

    if max_params is not None and "num_params" in d.columns:
        d["num_params"] = pd.to_numeric(d["num_params"], errors="coerce")
        d = d[d["num_params"] <= max_params].copy()

    return d


def choose_metric_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

## 2. Experimento 0 — dados reais de mercado primeiro

Esta seção apenas descreve o dataset real e escolhe o contrato com mais observações.  
A comparação com modelos vem depois.

In [ ]:
from utils.get_data import get_data

data_yf, opt = get_data("NDX")

opt = opt.copy()
opt["date"] = pd.to_datetime(opt["date"])

px = data_yf.reset_index().copy()
px["Date"] = pd.to_datetime(px["Date"])

market_all = opt.merge(px, left_on="date", right_on="Date", how="inner")

market_all = market_all[[
    "date",
    "Close",
    "current_time",
    "strike_price",
    "best_bid",
    "best_offer",
    "ticker",
    "impl_volatility",
]].copy()

market_all = market_all.rename(columns={"Close": "spot_price"})

# Ajustes do projeto
market_all["strike_price"] = pd.to_numeric(market_all["strike_price"], errors="coerce") / 1000.0
market_all["best_bid"] = pd.to_numeric(market_all["best_bid"], errors="coerce")
market_all["best_offer"] = pd.to_numeric(market_all["best_offer"], errors="coerce")
market_all["market_price"] = (market_all["best_bid"] + market_all["best_offer"]) / 2.0
market_all["tau"] = pd.to_numeric(market_all["current_time"], errors="coerce")
if market_all["tau"].median() > 5:
    market_all["tau"] = market_all["tau"] / 365.0

market_all["impl_volatility"] = pd.to_numeric(market_all["impl_volatility"], errors="coerce")
if market_all["impl_volatility"].median() > 3:
    market_all["impl_volatility"] = market_all["impl_volatility"] / 100.0

market_all["r"] = TRAIN_R
market_all["moneyness"] = market_all["spot_price"] / market_all["strike_price"]

market_all = market_all.dropna(subset=["date", "spot_price", "strike_price", "market_price", "tau", "impl_volatility"]).copy()
market_all = market_all.sort_values(["ticker", "date"]).reset_index(drop=True)

ticker_counts = market_all["ticker"].value_counts().reset_index()
ticker_counts.columns = ["ticker", "n_obs"]
display(ticker_counts.head(20))

ticker_principal = ticker_counts.iloc[0]["ticker"]
market = market_all[market_all["ticker"] == ticker_principal].copy()
market = market.sort_values("date").reset_index(drop=True)

print("Ticker selecionado:", ticker_principal)
print("N:", len(market))
print("K mediano:", market["strike_price"].median())
print("Spot min/max:", market["spot_price"].min(), market["spot_price"].max())
print("Tau min/max:", market["tau"].min(), market["tau"].max())
print("Moneyness min/max:", market["moneyness"].min(), market["moneyness"].max())

market.to_csv(TAB_DIR / "00_market_selected_clean.csv", index=False)
display(market.head())

In [ ]:
# Plots iniciais de mercado

fig, ax = plt.subplots(figsize=(7.2, 4.0))
ax.plot(market["date"], market["spot_price"], "o-", color=COLORS["Black-Scholes IV"], markersize=4)
ax.set_xlabel("Date")
ax.set_ylabel("Spot price")
ax.set_title("NDX selected contract: underlying spot")
savefig("00a_market_spot_time.png")

fig, ax = plt.subplots(figsize=(7.2, 4.0))
ax.plot(market["date"], market["market_price"], "ko-", markersize=4)
ax.set_xlabel("Date")
ax.set_ylabel("Option midpoint")
ax.set_title("NDX selected contract: market option price")
savefig("00b_market_price_time.png")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
axes[0].hist(market["moneyness"], bins=18, color="#5E81B5", edgecolor="black")
axes[0].set_title("Moneyness")
axes[0].set_xlabel("S/K")
axes[1].hist(market["tau"], bins=18, color="#E19C24", edgecolor="black")
axes[1].set_title("Time to maturity")
axes[1].set_xlabel("tau")
axes[2].hist(market["impl_volatility"], bins=18, color="#8FB131", edgecolor="black")
axes[2].set_title("Implied volatility")
axes[2].set_xlabel("sigma")
savefig("00c_market_histograms.png")

## 3. Carregar Greeks, sumários e aplicar filtros

Rode antes:

```bash
python calculate_greeks_from_runs.py
```

Este notebook usa `experimentos_pinn/greeks/greeks_summary.csv`.

In [ ]:
GREEKS_SUMMARY = RESULTS_DIR / "greeks" / "greeks_summary.csv"
if not GREEKS_SUMMARY.exists():
    raise FileNotFoundError(f"Não encontrei {GREEKS_SUMMARY}. Rode python calculate_greeks_from_runs.py")

greeks = pd.read_csv(GREEKS_SUMMARY)
greeks = to_numeric_if_possible(greeks)

if "family" not in greeks.columns:
    greeks["family"] = greeks.apply(normalize_family, axis=1)
else:
    greeks["family"] = greeks["family"].apply(normalize_family)

# Criar RMSE caso só exista MSE
for key in ["V", "delta", "gamma", "theta"]:
    rmse = f"RMSE_{key}_global"
    mse = f"MSE_{key}_global"
    mae = f"MAE_{key}_global"
    if rmse not in greeks.columns and mse in greeks.columns:
        greeks[rmse] = np.sqrt(pd.to_numeric(greeks[mse], errors="coerce"))
    if mae in greeks.columns:
        greeks[mae] = pd.to_numeric(greeks[mae], errors="coerce")

if "MAE_V_global" not in greeks.columns:
    raise ValueError("greeks_summary.csv precisa ter MAE_V_global para o teste de 1 dólar.")

greeks = apply_model_selection(greeks)
greeks = greeks.dropna(subset=["MAE_V_global"]).copy()

if "num_params" in greeks.columns:
    greeks["num_params"] = pd.to_numeric(greeks["num_params"], errors="coerce")
else:
    greeks["num_params"] = np.nan

print("Greeks após filtros:", greeks.shape)
display(greeks[[c for c in ["family", "model_type", "run_id", "MAE_V_global", "RMSE_V_global", "num_params", "hidden", "blocks", "n_qubits", "n_layers", "seed"] if c in greeks.columns]].head(20))

greeks.to_csv(TAB_DIR / "01_greeks_filtered.csv", index=False)

In [ ]:
# Selecionar melhor modelo por família para curvas 1D
best_models = (
    greeks
    .sort_values(["family", "MAE_V_global", "num_params"])
    .groupby("family", as_index=False)
    .first()
)

display(best_models[[c for c in ["family", "model_type", "run_id", "MAE_V_global", "RMSE_V_global", "num_params", "hidden", "blocks", "n_qubits", "n_layers", "seed"] if c in best_models.columns]])

best_models.to_csv(TAB_DIR / "02_best_models_by_family.csv", index=False)

## 4. Experimento 1 — Losses

Aqui entram as curvas de loss.  
A ideia importante: não usar tempo de cálculo. Usamos apenas **época/checkpoint** e **loss final/anytime**.

In [ ]:
def find_loss_file(row):
    candidates = []
    for c in ["loss_history_path", "loss_path"]:
        if c in row.index and pd.notna(row[c]):
            candidates.append(Path(str(row[c])))

    run_id = str(row.get("run_id", ""))
    model_type = str(row.get("model_type", row.get("family", "")))
    family = str(row.get("family", ""))

    for mt in [model_type, family, family.lower(), model_type.lower()]:
        candidates.append(RESULTS_DIR / "runs" / mt / run_id / "losses" / "loss_history_full.json")
        candidates.append(RESULTS_DIR / "runs" / mt / run_id / "loss_history.json")
        candidates.append(RESULTS_DIR / "runs" / mt / run_id / "metadata" / "loss_history_full.json")

    matches = list((RESULTS_DIR / "runs").glob(f"**/{run_id}/**/loss_history_full.json"))
    candidates.extend(matches)

    for p in candidates:
        if p.exists():
            return p
    return None


def normalize_loss_json(obj):
    if isinstance(obj, dict):
        for key in ["history", "loss_history", "losses"]:
            if key in obj:
                obj = obj[key]
                break

    if isinstance(obj, list):
        if len(obj) == 0:
            return pd.DataFrame()
        if isinstance(obj[0], dict):
            return pd.DataFrame(obj)
        return pd.DataFrame({"total_loss": obj})

    if isinstance(obj, dict):
        lens = {k: len(v) for k, v in obj.items() if isinstance(v, (list, tuple, np.ndarray))}
        if lens:
            n = max(set(lens.values()), key=list(lens.values()).count)
            data = {k: v for k, v in obj.items() if isinstance(v, (list, tuple, np.ndarray)) and len(v) == n}
            return pd.DataFrame(data)
        return pd.DataFrame([obj])

    return pd.DataFrame()


def load_losses_for_greeks(greeks_df):
    rows = []
    for _, row in greeks_df.iterrows():
        p = find_loss_file(row)
        if p is None:
            continue
        try:
            obj = json.loads(p.read_text())
            h = normalize_loss_json(obj)
            if h.empty:
                continue
            h = to_numeric_if_possible(h)
            if "epoch" not in h.columns:
                h["epoch"] = np.arange(len(h))

            rename = {}
            for c in h.columns:
                cl = c.lower()
                if cl in ["loss", "total", "total_loss", "loss_total", "train_loss"]:
                    rename[c] = "total_loss"
                elif "pde" in cl or "residual" in cl or "physics" in cl:
                    rename[c] = "pde_loss"
                elif "terminal" in cl or "initial" in cl or "ivp" in cl:
                    rename[c] = "terminal_loss"
                elif "boundary_0" in cl or "b0" in cl or "smin" in cl:
                    rename[c] = "boundary_0_loss"
                elif "boundary_max" in cl or "bmax" in cl or "smax" in cl:
                    rename[c] = "boundary_max_loss"
                elif "boundary" in cl or "bvp" in cl:
                    rename[c] = "boundary_loss"
            h = h.rename(columns=rename)

            h["run_id"] = str(row["run_id"])
            h["family"] = str(row["family"])
            h["model_type"] = str(row.get("model_type", ""))
            h["num_params"] = row.get("num_params", np.nan)
            h["MAE_V_global"] = row.get("MAE_V_global", np.nan)
            h["loss_file"] = str(p)
            rows.append(h)
        except Exception as e:
            print("Falha lendo loss:", row.get("run_id"), e)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

loss_long = load_losses_for_greeks(greeks)

print("loss_long:", loss_long.shape)
if len(loss_long):
    display(loss_long.head())
    loss_long.to_csv(TAB_DIR / "03_loss_long_filtered.csv", index=False)

In [ ]:
# Loss total das melhores runs por família

if len(loss_long) and "total_loss" in loss_long.columns:
    best_ids = set(best_models["run_id"].astype(str))
    sub = loss_long[loss_long["run_id"].astype(str).isin(best_ids)].copy()

    plt.figure(figsize=(7.4, 4.2))
    for run_id, g in sub.groupby("run_id"):
        g = g.sort_values("epoch")
        fam = g["family"].iloc[0]
        y = pd.to_numeric(g["total_loss"], errors="coerce").rolling(25, min_periods=1).median()
        plt.plot(g["epoch"], y, label=f"{fam}")

    plt.yscale("log")
    plt.xlabel("Epoch")
    plt.ylabel("Total loss")
    plt.title("Loss total das melhores runs por família")
    plt.legend()
    savefig("01a_best_runs_total_loss.png")
else:
    print("Loss total não encontrada.")

In [ ]:
# Loss por componente das melhores runs

component_cols = [c for c in ["pde_loss", "terminal_loss", "boundary_0_loss", "boundary_max_loss", "boundary_loss"] if c in loss_long.columns]

if len(loss_long) and component_cols:
    best_ids = list(best_models["run_id"].astype(str))
    n = len(best_ids)
    fig, axes = plt.subplots(n, 1, figsize=(7.4, max(3.2, 2.8*n)), sharex=True)
    if n == 1:
        axes = [axes]

    for ax, run_id in zip(axes, best_ids):
        g = loss_long[loss_long["run_id"].astype(str) == run_id].sort_values("epoch")
        fam = g["family"].iloc[0] if len(g) else ""
        for c in component_cols:
            y = pd.to_numeric(g[c], errors="coerce").rolling(25, min_periods=1).median()
            if y.notna().sum():
                ax.plot(g["epoch"], y, label=c)
        ax.set_yscale("log")
        ax.set_ylabel("Loss")
        ax.set_title(f"{fam} | {run_id}")
        ax.legend(ncol=2, fontsize=8)

    axes[-1].set_xlabel("Epoch")
    savefig("01b_best_runs_loss_components.png")
else:
    print("Componentes de loss não encontrados.")

In [ ]:
# Análise "anytime": quem tem menor loss em checkpoints?
# Use como diagnóstico, não como comparação principal se os checkpoints não foram definidos antes.

if len(loss_long) and "total_loss" in loss_long.columns:
    rows = []
    for run_id, g in loss_long.groupby("run_id"):
        g = g.sort_values("epoch")
        fam = g["family"].iloc[0]
        params = g["num_params"].iloc[0]
        mae = g["MAE_V_global"].iloc[0]
        max_epoch = g["epoch"].max()
        for ck in LOSS_CHECKPOINTS:
            gck = g[g["epoch"] <= ck]
            if len(gck) == 0:
                continue
            last = gck.iloc[-1]
            rows.append({
                "run_id": run_id,
                "family": fam,
                "num_params": params,
                "synthetic_MAE_V_global": mae,
                "checkpoint": ck,
                "epoch_observed": last["epoch"],
                "total_loss_at_checkpoint": last["total_loss"],
                "reached_checkpoint": max_epoch >= ck,
            })

    loss_checkpoints = pd.DataFrame(rows)
    loss_checkpoints.to_csv(TAB_DIR / "03b_loss_checkpoints.csv", index=False)
    display(loss_checkpoints.head())

    # Mediana por família no checkpoint
    loss_ck_summary = (
        loss_checkpoints
        .groupby(["checkpoint", "family"])
        .agg(
            n_runs=("run_id", "count"),
            median_loss=("total_loss_at_checkpoint", "median"),
            best_loss=("total_loss_at_checkpoint", "min"),
            median_synthetic_MAE=("synthetic_MAE_V_global", "median"),
        )
        .reset_index()
    )
    display(loss_ck_summary)
    loss_ck_summary.to_csv(TAB_DIR / "03c_loss_checkpoint_summary.csv", index=False)

    plt.figure(figsize=(7.4, 4.3))
    for fam in FAMILIES:
        g = loss_ck_summary[loss_ck_summary["family"] == fam]
        if len(g):
            plt.plot(g["checkpoint"], g["median_loss"], "o-", label=fam, color=COLORS.get(fam))
    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel("Epoch checkpoint")
    plt.ylabel("Median total loss")
    plt.title("Anytime loss diagnostic by family")
    plt.legend()
    savefig("01c_anytime_median_loss_by_family.png")
else:
    print("Sem loss para análise anytime.")

## 5. Experimento 2 — taxa de convergência sintética: erro médio menor que 1 dólar

Agora a interpretação correta: **taxa de sucesso**.

```text
success rate = % das runs com MAE_V_global < 1 dólar sintético
```

In [ ]:
conv = greeks.copy()
conv = conv[conv["family"].isin(FAMILIES)].copy()
conv["MAE_V_global"] = pd.to_numeric(conv["MAE_V_global"], errors="coerce")
conv["num_params"] = pd.to_numeric(conv["num_params"], errors="coerce")
conv = conv.dropna(subset=["MAE_V_global"]).copy()

conv["success_lt_1_dollar"] = conv["MAE_V_global"] < SYNTHETIC_ONE_DOLLAR_THRESHOLD

convergence_by_family = (
    conv
    .groupby("family")
    .agg(
        n_runs=("run_id", "count"),
        n_success=("success_lt_1_dollar", "sum"),
        success_rate=("success_lt_1_dollar", "mean"),
        best_MAE=("MAE_V_global", "min"),
        median_MAE=("MAE_V_global", "median"),
        mean_MAE=("MAE_V_global", "mean"),
        std_MAE=("MAE_V_global", "std"),
        median_params=("num_params", "median"),
    )
    .reset_index()
)

convergence_by_family["success_rate_pct"] = 100 * convergence_by_family["success_rate"]
convergence_by_family = convergence_by_family.sort_values(["success_rate_pct", "median_MAE"], ascending=[False, True])

display(convergence_by_family)
convergence_by_family.to_csv(TAB_DIR / "04_synthetic_convergence_rate_lt_1_dollar_by_family.csv", index=False)

In [ ]:
plt.figure(figsize=(6.4, 4.1))
plot_df = convergence_by_family.copy()

plt.bar(
    plot_df["family"],
    plot_df["success_rate_pct"],
    color=[COLORS.get(f, "#777777") for f in plot_df["family"]],
)
plt.ylim(0, 100)
plt.ylabel("Success rate (%)")
plt.xlabel("Family")
plt.title(r"Synthetic success rate: $\mathrm{MAE}_V < 1$ dollar")

for i, row in plot_df.reset_index(drop=True).iterrows():
    plt.text(i, row["success_rate_pct"] + 2, f"{int(row['n_success'])}/{int(row['n_runs'])}",
             ha="center", va="bottom", fontsize=10)

savefig("02a_synthetic_success_rate_by_family.png")

## 6. Experimento 3 — comparação justa por orçamento de parâmetros

Esta é a análise principal no estilo Trahan: comparar modelos com **orçamento parecido de parâmetros**.

In [ ]:
conv_budget = conv.dropna(subset=["num_params", "MAE_V_global"]).copy()
conv_budget["param_budget"] = pd.cut(
    conv_budget["num_params"],
    bins=PARAM_BINS,
    labels=PARAM_LABELS,
    include_lowest=True,
    right=True,
)

budget_success = (
    conv_budget
    .groupby(["param_budget", "family"], observed=False)
    .agg(
        n_runs=("run_id", "count"),
        n_success=("success_lt_1_dollar", "sum"),
        success_rate=("success_lt_1_dollar", "mean"),
        best_MAE=("MAE_V_global", "min"),
        median_MAE=("MAE_V_global", "median"),
        mean_MAE=("MAE_V_global", "mean"),
        std_MAE=("MAE_V_global", "std"),
        median_params=("num_params", "median"),
    )
    .reset_index()
)

budget_success["success_rate_pct"] = 100 * budget_success["success_rate"]
budget_success = budget_success.sort_values(["param_budget", "family"])

display(budget_success)
budget_success.to_csv(TAB_DIR / "05_synthetic_success_rate_by_parameter_budget.csv", index=False)

In [ ]:
# Plot: taxa de sucesso por faixa de parâmetros

pivot_success = budget_success.pivot(index="param_budget", columns="family", values="success_rate_pct")
pivot_n = budget_success.pivot(index="param_budget", columns="family", values="n_runs")

ax = pivot_success.plot(
    kind="bar",
    figsize=(9.0, 4.8),
    width=0.82,
    color=[COLORS.get(c, None) for c in pivot_success.columns],
)
ax.set_ylim(0, 100)
ax.set_xlabel("Parameter budget")
ax.set_ylabel("Success rate (%)")
ax.set_title(r"Success rate by parameter budget: $\mathrm{MAE}_V < 1$ dollar")
ax.grid(True, axis="y", color="0.85")

for container, fam in zip(ax.containers, pivot_success.columns):
    for bar, budget in zip(container, pivot_success.index):
        if fam in pivot_n.columns:
            n_val = pivot_n.loc[budget, fam]
            if pd.notna(n_val) and n_val > 0 and np.isfinite(bar.get_height()):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                        f"n={int(n_val)}", ha="center", va="bottom", fontsize=7, rotation=90)

plt.xticks(rotation=35, ha="right")
savefig("03a_success_rate_by_parameter_budget.png")

# Plot: MAE mediano por faixa de parâmetros
pivot_median = budget_success.pivot(index="param_budget", columns="family", values="median_MAE")
ax = pivot_median.plot(
    kind="bar",
    figsize=(9.0, 4.8),
    width=0.82,
    color=[COLORS.get(c, None) for c in pivot_median.columns],
)
ax.axhline(SYNTHETIC_ONE_DOLLAR_THRESHOLD, color="black", linestyle="--", linewidth=1.2, label="1 dollar")
ax.set_yscale("log")
ax.set_xlabel("Parameter budget")
ax.set_ylabel(r"Median synthetic $\mathrm{MAE}_V$")
ax.set_title("Median synthetic error by parameter budget")
ax.grid(True, which="both", axis="y", color="0.85")
ax.legend()
plt.xticks(rotation=35, ha="right")
savefig("03b_median_mae_by_parameter_budget.png")

# Plot: melhor MAE por faixa
pivot_best = budget_success.pivot(index="param_budget", columns="family", values="best_MAE")
ax = pivot_best.plot(
    kind="bar",
    figsize=(9.0, 4.8),
    width=0.82,
    color=[COLORS.get(c, None) for c in pivot_best.columns],
)
ax.axhline(SYNTHETIC_ONE_DOLLAR_THRESHOLD, color="black", linestyle="--", linewidth=1.2, label="1 dollar")
ax.set_yscale("log")
ax.set_xlabel("Parameter budget")
ax.set_ylabel(r"Best synthetic $\mathrm{MAE}_V$")
ax.set_title("Best synthetic error by parameter budget")
ax.grid(True, which="both", axis="y", color="0.85")
ax.legend()
plt.xticks(rotation=35, ha="right")
savefig("03c_best_mae_by_parameter_budget.png")

## 7. Experimento 4 — curvas sintéticas 1D: preço e erro

Usa os melhores modelos selecionados por família.

In [ ]:
# Importar loader de modelos
from calculate_greeks_from_runs import load_model
import torch as tc

def predict_model_curve(model, S, t):
    model.eval()

    S_t = tc.tensor((S / TRAIN_S_MAX).reshape(-1, 1), dtype=tc.float32, requires_grad=True)
    t_t = tc.tensor((np.full_like(S, t) / TRAIN_T).reshape(-1, 1), dtype=tc.float32, requires_grad=True)
    x = tc.cat([S_t, t_t], dim=1)

    V_norm = model(x).reshape(-1, 1)
    ones = tc.ones_like(V_norm)

    dVdS_norm = tc.autograd.grad(V_norm, S_t, grad_outputs=ones, create_graph=True, retain_graph=True)[0]
    dVdt_norm = tc.autograd.grad(V_norm, t_t, grad_outputs=ones, create_graph=True, retain_graph=True)[0]
    d2VdS2_norm = tc.autograd.grad(dVdS_norm, S_t, grad_outputs=tc.ones_like(dVdS_norm), create_graph=False, retain_graph=False)[0]

    return {
        "V": V_norm.detach().numpy().reshape(-1) * TRAIN_V_MAX,
        "delta": dVdS_norm.detach().numpy().reshape(-1) * (TRAIN_V_MAX / TRAIN_S_MAX),
        "gamma": d2VdS2_norm.detach().numpy().reshape(-1) * (TRAIN_V_MAX / TRAIN_S_MAX**2),
        "theta": dVdt_norm.detach().numpy().reshape(-1) * (TRAIN_V_MAX / TRAIN_T),
    }

S_grid = np.linspace(0.1, TRAIN_S_MAX, 300)
t_cuts = [0.00, 0.50, 0.90, 0.99]

loaded_models = {}
for _, row in best_models.iterrows():
    fam = row["family"]
    run_dir = RESULTS_DIR / "runs" / str(row["model_type"]) / str(row["run_id"])
    print("Carregando:", fam, run_dir)
    model, cfg, state_path = load_model(run_dir, device=DEVICE)
    loaded_models[fam] = model

synthetic_rows = []
for t0 in t_cuts:
    V_BS, delta_BS, gamma_BS, theta_BS = bs_call_greeks_t(S_grid, np.full_like(S_grid, t0))
    for fam, model in loaded_models.items():
        pred = predict_model_curve(model, S_grid, t0)
        tmp = pd.DataFrame({
            "family": fam,
            "S": S_grid,
            "t": t0,
            "moneyness": S_grid / TRAIN_K,
            "V_BS": V_BS,
            "delta_BS": delta_BS,
            "gamma_BS": gamma_BS,
            "theta_BS": theta_BS,
            "V_pred": pred["V"],
            "delta_pred": pred["delta"],
            "gamma_pred": pred["gamma"],
            "theta_pred": pred["theta"],
        })
        synthetic_rows.append(tmp)

synthetic_pred = pd.concat(synthetic_rows, ignore_index=True)
synthetic_pred["moneyness_region"] = pd.cut(
    synthetic_pred["moneyness"],
    bins=[-np.inf, 0.97, 1.03, np.inf],
    labels=["OTM", "ATM", "ITM"],
)
synthetic_pred["abs_error_V"] = np.abs(synthetic_pred["V_pred"] - synthetic_pred["V_BS"])
synthetic_pred.to_csv(TAB_DIR / "06_synthetic_curves_predictions.csv", index=False)
display(synthetic_pred.head())

In [ ]:
# Curvas sintéticas de preço V(S) por corte de tempo

for t0 in t_cuts:
    sub = synthetic_pred[synthetic_pred["t"] == t0].copy()
    plt.figure(figsize=(7.2, 4.2))
    bs = sub.drop_duplicates("S").sort_values("S")
    plt.plot(bs["S"], bs["V_BS"], "k-", linewidth=2.2, label="Black-Scholes")
    for fam in FAMILIES:
        g = sub[sub["family"] == fam].sort_values("S")
        if len(g):
            plt.plot(g["S"], g["V_pred"], linewidth=1.9, color=COLORS.get(fam), label=fam)
    plt.axvline(TRAIN_K, color="0.35", linestyle="--", linewidth=1)
    plt.xlabel(r"$S$")
    plt.ylabel(r"$V(S,t)$")
    plt.title(fr"Synthetic price curve at $t={t0:.2f}$")
    plt.legend()
    savefig(f"04a_synthetic_price_curve_t_{t0:.2f}.png")

# Erro absoluto de preço
for t0 in t_cuts:
    sub = synthetic_pred[synthetic_pred["t"] == t0].copy()
    plt.figure(figsize=(7.2, 4.2))
    for fam in FAMILIES:
        g = sub[sub["family"] == fam].sort_values("S")
        if len(g):
            plt.plot(g["S"], np.abs(g["V_pred"] - g["V_BS"]), linewidth=1.9, color=COLORS.get(fam), label=fam)
    plt.axhline(SYNTHETIC_ONE_DOLLAR_THRESHOLD, color="black", linestyle="--", linewidth=1.2, label="1 dollar")
    plt.axvline(TRAIN_K, color="0.35", linestyle="--", linewidth=1)
    plt.yscale("log")
    plt.xlabel(r"$S$")
    plt.ylabel(r"$|V_{pred}-V_{BS}|$")
    plt.title(fr"Synthetic price absolute error at $t={t0:.2f}$")
    plt.legend()
    savefig(f"04b_synthetic_price_abs_error_t_{t0:.2f}.png")

## 8. Experimento 5 — Greeks sintéticas 1D

In [ ]:
for greek in ["delta", "gamma", "theta"]:
    for t0 in t_cuts:
        sub = synthetic_pred[synthetic_pred["t"] == t0].copy()
        plt.figure(figsize=(7.2, 4.2))
        bs = sub.drop_duplicates("S").sort_values("S")
        plt.plot(bs["S"], bs[f"{greek}_BS"], "k-", linewidth=2.2, label=f"BS {greek}")
        for fam in FAMILIES:
            g = sub[sub["family"] == fam].sort_values("S")
            if len(g):
                plt.plot(g["S"], g[f"{greek}_pred"], linewidth=1.8, color=COLORS.get(fam), label=fam)
        plt.axvline(TRAIN_K, color="0.35", linestyle="--", linewidth=1)
        plt.xlabel(r"$S$")
        plt.ylabel(greek)
        plt.title(fr"Synthetic {greek} at $t={t0:.2f}$")
        plt.legend()
        savefig(f"05a_synthetic_{greek}_curve_t_{t0:.2f}.png")

## 9. Experimento 6 — mercado real 1D com normalização por strike

Aqui o mercado é reescalado para a escala sintética do treino:

```text
scale = K_train / K_market
S_sim = S_market * scale
V_sim = V_market * scale
```

A saída volta para dólares reais por:

```text
V_market_pred = V_norm * V_max / scale
```

In [ ]:
def predict_model_on_market(model, df_market, prefix, scale):
    model.eval()

    S_sim = df_market["spot_price"].to_numpy(float) * scale
    tau_model = df_market["tau"].clip(0, TRAIN_T).to_numpy(float)
    t_model = (TRAIN_T - tau_model).clip(0, TRAIN_T)

    S_norm = tc.tensor((S_sim / TRAIN_S_MAX).reshape(-1, 1), dtype=tc.float32, requires_grad=True)
    t_norm = tc.tensor((t_model / TRAIN_T).reshape(-1, 1), dtype=tc.float32, requires_grad=True)
    x = tc.cat([S_norm, t_norm], dim=1)

    V_norm = model(x).reshape(-1, 1)
    ones = tc.ones_like(V_norm)

    dVdS_norm = tc.autograd.grad(V_norm, S_norm, grad_outputs=ones, create_graph=True, retain_graph=True)[0]
    dVdt_norm = tc.autograd.grad(V_norm, t_norm, grad_outputs=ones, create_graph=True, retain_graph=True)[0]
    d2VdS2_norm = tc.autograd.grad(dVdS_norm, S_norm, grad_outputs=tc.ones_like(dVdS_norm), create_graph=False, retain_graph=False)[0]

    V_sim = V_norm.detach().numpy().reshape(-1) * TRAIN_V_MAX
    delta_sim = dVdS_norm.detach().numpy().reshape(-1) * (TRAIN_V_MAX / TRAIN_S_MAX)
    gamma_sim = d2VdS2_norm.detach().numpy().reshape(-1) * (TRAIN_V_MAX / TRAIN_S_MAX**2)
    theta_sim = dVdt_norm.detach().numpy().reshape(-1) * (TRAIN_V_MAX / TRAIN_T)

    out = df_market.copy()
    out[f"{prefix}_price"] = V_sim / scale
    out[f"{prefix}_delta"] = delta_sim
    out[f"{prefix}_gamma"] = gamma_sim * scale
    out[f"{prefix}_theta"] = theta_sim / scale
    return out[[f"{prefix}_price", f"{prefix}_delta", f"{prefix}_gamma", f"{prefix}_theta"]]

market_pred = market.copy()
K_market = float(market_pred["strike_price"].median())
scale = TRAIN_K / K_market

market_pred["S_sim"] = market_pred["spot_price"] * scale
market_pred["K_sim"] = market_pred["strike_price"] * scale
market_pred["market_price_sim"] = market_pred["market_price"] * scale
market_pred["S_norm_check"] = market_pred["S_sim"] / TRAIN_S_MAX
market_pred["tau_model"] = market_pred["tau"].clip(0, TRAIN_T)
market_pred["t_model"] = (TRAIN_T - market_pred["tau_model"]).clip(0, TRAIN_T)
market_pred["t_norm_check"] = market_pred["t_model"] / TRAIN_T

print("Conversão mercado -> simulação")
print("K_market:", K_market)
print("scale = K_train / K_market:", scale)
print("1 dólar real equivale a", scale, "unidades simuladas")
print("S_norm min/max:", market_pred["S_norm_check"].min(), market_pred["S_norm_check"].max())
print("t_norm min/max:", market_pred["t_norm_check"].min(), market_pred["t_norm_check"].max())

# Benchmarks BS
(
    market_pred["bs_iv_price"],
    market_pred["bs_iv_delta"],
    market_pred["bs_iv_gamma"],
    market_pred["bs_iv_theta"],
) = bs_call_greeks_tau(
    market_pred["spot_price"],
    market_pred["tau_model"],
    market_pred["strike_price"],
    TRAIN_R,
    market_pred["impl_volatility"],
)

(
    market_pred["bs_train_price"],
    market_pred["bs_train_delta"],
    market_pred["bs_train_gamma"],
    market_pred["bs_train_theta"],
) = bs_call_greeks_tau(
    market_pred["spot_price"],
    market_pred["tau_model"],
    market_pred["strike_price"],
    TRAIN_R,
    TRAIN_SIGMA,
)

for fam, model in loaded_models.items():
    pred_cols = predict_model_on_market(model, market_pred, fam, scale)
    market_pred = pd.concat([market_pred, pred_cols], axis=1)

market_pred.to_csv(TAB_DIR / "07_market_predictions_normalized.csv", index=False)
display(market_pred.head())

In [ ]:
# Mercado: curvas temporais e contra spot

plt.figure(figsize=(7.5, 4.4))
plt.plot(market_pred["date"], market_pred["market_price"], "ko", markersize=4, label="Market")
plt.plot(market_pred["date"], market_pred["bs_iv_price"], "-", linewidth=2, color=COLORS["Black-Scholes IV"], label="BS IV")
plt.plot(market_pred["date"], market_pred["bs_train_price"], "--", linewidth=2, color=COLORS["Black-Scholes train sigma"], label="BS train sigma")
for fam in FAMILIES:
    col = f"{fam}_price"
    if col in market_pred.columns:
        plt.plot(market_pred["date"], market_pred[col], linewidth=1.9, color=COLORS.get(fam), label=fam)
plt.xlabel("Date")
plt.ylabel("Option price")
plt.title("Real market 1D curve: normalized model input")
plt.legend(ncol=2)
savefig("06a_market_price_time_models.png")

spot_sorted = market_pred.sort_values("spot_price")
plt.figure(figsize=(7.5, 4.4))
plt.plot(spot_sorted["spot_price"], spot_sorted["market_price"], "ko", markersize=4, label="Market")
plt.plot(spot_sorted["spot_price"], spot_sorted["bs_iv_price"], "-", linewidth=2, color=COLORS["Black-Scholes IV"], label="BS IV")
plt.plot(spot_sorted["spot_price"], spot_sorted["bs_train_price"], "--", linewidth=2, color=COLORS["Black-Scholes train sigma"], label="BS train sigma")
for fam in FAMILIES:
    col = f"{fam}_price"
    if col in spot_sorted.columns:
        plt.plot(spot_sorted["spot_price"], spot_sorted[col], linewidth=1.9, color=COLORS.get(fam), label=fam)
plt.xlabel(r"Spot $S$")
plt.ylabel("Option price")
plt.title("Real market price curve against spot")
plt.legend(ncol=2)
savefig("06b_market_price_spot_models.png")

In [ ]:
# Métricas mercado global e por região

methods_price = {
    "Black-Scholes IV": "bs_iv_price",
    "Black-Scholes train sigma": "bs_train_price",
}
for fam in FAMILIES:
    col = f"{fam}_price"
    if col in market_pred.columns:
        methods_price[fam] = col

market_metric_rows = []
for name, col in methods_price.items():
    m = metrics(market_pred["market_price"], market_pred[col])
    m["method"] = name
    m["negative_prediction_share"] = np.mean(market_pred[col] < 0)
    market_metric_rows.append(m)

market_metrics_global = pd.DataFrame(market_metric_rows).set_index("method")
display(market_metrics_global)
market_metrics_global.to_csv(TAB_DIR / "08_market_metrics_global.csv")

market_pred["moneyness_region"] = pd.cut(
    market_pred["moneyness"], bins=[-np.inf, 0.97, 1.03, np.inf], labels=["OTM", "ATM", "ITM"]
)
market_pred["maturity_region"] = pd.qcut(
    market_pred["tau_model"], q=3, labels=["Short", "Medium", "Long"], duplicates="drop"
)

region_rows = []
for region_col in ["moneyness_region", "maturity_region"]:
    for region, g in market_pred.groupby(region_col):
        for name, col in methods_price.items():
            m = metrics(g["market_price"], g[col])
            m["region_type"] = region_col
            m["region"] = str(region)
            m["method"] = name
            region_rows.append(m)

market_metrics_region = pd.DataFrame(region_rows)
display(market_metrics_region)
market_metrics_region.to_csv(TAB_DIR / "09_market_metrics_by_region.csv", index=False)

In [ ]:
# Plot RMSE mercado por região

for region_type in ["moneyness_region", "maturity_region"]:
    tmp = market_metrics_region[market_metrics_region["region_type"] == region_type]
    pivot = tmp.pivot(index="region", columns="method", values="RMSE")
    ax = pivot.plot(kind="bar", figsize=(8.4, 4.4))
    ax.set_ylabel("RMSE vs market")
    ax.set_xlabel(region_type)
    ax.set_title(f"Market RMSE by {region_type}")
    plt.xticks(rotation=0)
    ax.grid(True, axis="y", color="0.85")
    savefig(f"06c_market_rmse_by_{region_type}.png")

## 10. Experimento 7 — Greeks no mercado

In [ ]:
for greek in ["delta", "gamma", "theta"]:
    plt.figure(figsize=(7.5, 4.4))
    plt.plot(market_pred["date"], market_pred[f"bs_iv_{greek}"], "-", linewidth=2, color=COLORS["Black-Scholes IV"], label=f"BS IV {greek}")
    plt.plot(market_pred["date"], market_pred[f"bs_train_{greek}"], "--", linewidth=2, color=COLORS["Black-Scholes train sigma"], label=f"BS train sigma {greek}")
    for fam in FAMILIES:
        col = f"{fam}_{greek}"
        if col in market_pred.columns:
            plt.plot(market_pred["date"], market_pred[col], linewidth=1.8, color=COLORS.get(fam), label=fam)
    plt.xlabel("Date")
    plt.ylabel(greek)
    plt.title(f"Market 1D {greek} curves")
    plt.legend(ncol=2, fontsize=8)
    savefig(f"07a_market_{greek}_curves.png")

# Métricas de Greeks contra BS IV
greek_metric_rows = []
for greek in ["delta", "gamma", "theta"]:
    true_col = f"bs_iv_{greek}"
    for fam in FAMILIES:
        pred_col = f"{fam}_{greek}"
        if pred_col in market_pred.columns:
            m = metrics(market_pred[true_col], market_pred[pred_col])
            m["greek"] = greek
            m["family"] = fam
            greek_metric_rows.append(m)

market_greek_metrics = pd.DataFrame(greek_metric_rows)
display(market_greek_metrics)
market_greek_metrics.to_csv(TAB_DIR / "10_market_greek_metrics_vs_bs_iv.csv", index=False)

## 11. Resumo final automático para interpretação

In [ ]:
summary_lines = []

summary_lines.append("## Resumo automático")
summary_lines.append("")
summary_lines.append(f"Ticker de mercado selecionado: `{ticker_principal}` com N={len(market)} pontos.")
summary_lines.append("")
summary_lines.append("### Taxa global de sucesso sintético")
summary_lines.append(convergence_by_family.to_markdown(index=False))
summary_lines.append("")
summary_lines.append("### Sucesso por orçamento de parâmetros")
summary_lines.append(budget_success.to_markdown(index=False))
summary_lines.append("")
summary_lines.append("### Métricas globais de mercado")
summary_lines.append(market_metrics_global.reset_index().to_markdown(index=False))
summary_lines.append("")

if len(loss_long) and "total_loss" in loss_long.columns:
    summary_lines.append("### Observação sobre losses")
    summary_lines.append(
        "A análise de loss por checkpoint é diagnóstica. "
        "Se a hipótese de parar cedo não foi definida antes, a comparação principal deve usar o orçamento completo "
        "ou reportar a curva completa/anytime, não escolher retrospectivamente o melhor checkpoint."
    )
    summary_lines.append("")

summary_text = "\n".join(summary_lines)
report_path = TAB_DIR / "RELATORIO_RESUMO_AUTOMATICO.md"
report_path.write_text(summary_text, encoding="utf-8")

from IPython.display import Markdown, display
display(Markdown(summary_text))
print("Relatório salvo em:", report_path)